# 🎯 Système de Recommandation d'Offres d'Emploi - Démonstration

Ce notebook démontre l'utilisation du système de recommandation intelligent basé sur l'IA pour les offres d'emploi dans le domaine de la Data.

## Technologies Utilisées

- **Sentence-BERT** : Embeddings sémantiques multilingues
- **FAISS** : Recherche vectorielle ultra-rapide
- **spaCy** : Extraction de compétences (NER)
- **scikit-learn** : Calculs de similarité

## 1. Initialisation du Système

In [ ]:
# Imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

# Importer notre système
from job_recommender import JobRecommender
from cv_parser import CVParser
from data_preprocessing import JobDataPreprocessor

print("✅ Imports réussis")

In [ ]:
# Initialiser le recommender
# Note: La première fois, cela prendra 5-10 minutes pour créer les embeddings
# Les fois suivantes, il chargera les embeddings sauvegardés (~10 secondes)

recommender = JobRecommender()

print(f"\n📊 Système chargé avec {len(recommender.jobs_df):,} offres d'emploi")

## 2. Exploration des Données

In [ ]:
# Aperçu des données
recommender.jobs_df.head()

In [ ]:
# Statistiques du système
stats = recommender.get_statistics()

print("="*80)
print("STATISTIQUES DU SYSTÈME")
print("="*80)
print(f"Total offres: {stats['total_jobs']:,}")
print(f"Entreprises uniques: {stats['unique_companies']:,}")
print(f"Localisations uniques: {stats['unique_locations']:,}")
print(f"Types de contrat uniques: {stats['unique_contract_types']:,}")
print(f"Compétences moyennes par offre: {stats['avg_skills_per_job']:.1f}")
print("\n📊 Top 10 Compétences:")
for skill, count in stats['top_10_skills']:
    print(f"  {skill}: {count:,}")


In [ ]:
# Visualisation des top compétences
top_skills = stats['top_10_skills']
skills_df = pd.DataFrame(top_skills, columns=['Compétence', 'Nombre'])

plt.figure(figsize=(12, 6))
sns.barplot(data=skills_df, x='Nombre', y='Compétence', palette='viridis')
plt.title('Top 10 des Compétences les Plus Demandées', fontsize=16, fontweight='bold')
plt.xlabel('Nombre d\'offres', fontsize=12)
plt.ylabel('Compétence', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des niveaux d'expérience
exp_dist = stats['experience_level_distribution']
exp_df = pd.DataFrame(list(exp_dist.items()), columns=['Niveau', 'Nombre'])

plt.figure(figsize=(10, 6))
sns.barplot(data=exp_df, x='Niveau', y='Nombre', palette='coolwarm')
plt.title('Distribution des Niveaux d\'Expérience', fontsize=16, fontweight='bold')
plt.xlabel('Niveau d\'expérience', fontsize=12)
plt.ylabel('Nombre d\'offres', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Test de Recommandation - Profil Data Scientist

In [ ]:
# Définir un profil candidat exemple
candidate_profile = """
Data Scientist passionné avec 3 ans d'expérience en Machine Learning et Deep Learning.
Expert en Python, TensorFlow, PyTorch, et scikit-learn.
Solides compétences en statistiques, mathématiques et modélisation prédictive.
Expérience avec les données massives (Spark, Hadoop) et le cloud (AWS, Azure).
Recherche un poste challengeant en région parisienne ou en remote.
"""

keywords = [
    'Python', 'Machine Learning', 'Deep Learning', 'TensorFlow', 'PyTorch',
    'scikit-learn', 'SQL', 'Pandas', 'NumPy', 'Statistics',
    'Spark', 'AWS', 'Data Science'
]

print("🔍 Profil Candidat:")
print(candidate_profile)
print(f"\n💼 Compétences: {', '.join(keywords)}")

In [ ]:
# Obtenir des recommandations
recommendations = recommender.recommend(
    candidate_profile=candidate_profile,
    keywords=keywords,
    location_preference="Paris",
    contract_type_preference="Full-time",
    top_k=10,
    min_score=0.0
)

print(f"\n✨ Trouvé {len(recommendations)} recommandations\n")

In [ ]:
# Afficher les résultats
for i, rec in enumerate(recommendations, 1):
    print(f"\n{'='*80}")
    print(f"{i}. {rec['title']}")
    print(f"{'='*80}")
    print(f"🏢 Entreprise: {rec['company']}")
    print(f"📍 Localisation: {rec['location']}")
    print(f"📄 Contrat: {rec['contract_type']}")
    print(f"💼 Type: {rec['work_type']}")
    print(f"🎯 Score: {rec['score']:.3f}")
    print(f"   - Similarité sémantique: {rec['semantic_similarity']:.3f}")
    print(f"   - Compétences matchées: {rec['skills_match_count']}")
    print(f"   - Ratio de match: {rec['skills_match_ratio']:.2%}")
    print(f"\n💡 Compétences requises:")
    print(f"   {', '.join(rec['skills'][:10])}")
    if len(rec['skills']) > 10:
        print(f"   ... et {len(rec['skills']) - 10} autres")
    print(f"\n🔗 Lien: {rec['job_url'][:80]}...")

In [ ]:
# Créer un DataFrame pour analyse
recs_df = pd.DataFrame([{
    'Rang': i,
    'Titre': rec['title'][:40],
    'Entreprise': rec['company'][:25],
    'Localisation': rec['location'][:20],
    'Score': rec['score'],
    'Similarité': rec['semantic_similarity'],
    'Skills Match': rec['skills_match_count']
} for i, rec in enumerate(recommendations, 1)])

recs_df

In [ ]:
# Visualisation des scores
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Score final
axes[0].barh(recs_df['Rang'], recs_df['Score'], color='steelblue')
axes[0].set_xlabel('Score Final', fontsize=12)
axes[0].set_ylabel('Rang', fontsize=12)
axes[0].set_title('Scores Finaux des Recommandations', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()

# Similarité sémantique vs Skills match
axes[1].scatter(recs_df['Similarité'], recs_df['Skills Match'], 
                s=recs_df['Score']*500, alpha=0.6, c=recs_df['Score'], 
                cmap='viridis', edgecolors='black')
axes[1].set_xlabel('Similarité Sémantique', fontsize=12)
axes[1].set_ylabel('Nombre de Compétences Matchées', fontsize=12)
axes[1].set_title('Similarité vs Compétences (taille = score final)', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Test avec un Autre Profil - Data Engineer

In [ ]:
# Profil Data Engineer
de_profile = """
Data Engineer senior avec 5+ ans d'expérience en construction de pipelines de données.
Expert en Spark, Hadoop, Kafka, Airflow.
Solides compétences en Python, SQL, et cloud (AWS, GCP).
Expérience avec Databricks, Snowflake, et architectures Data Lake.
"""

de_keywords = [
    'Python', 'Spark', 'Hadoop', 'Kafka', 'Airflow',
    'SQL', 'ETL', 'AWS', 'GCP', 'Databricks', 'Snowflake'
]

de_recommendations = recommender.recommend(
    candidate_profile=de_profile,
    keywords=de_keywords,
    experience_level='senior',
    top_k=5
)

print(f"\n🔧 Top 5 offres pour Data Engineer Senior:\n")
for i, rec in enumerate(de_recommendations, 1):
    print(f"{i}. {rec['title']} - {rec['company']}")
    print(f"   Score: {rec['score']:.3f} | Skills: {rec['skills_match_count']}")
    print()

## 5. Test de Recherche d'Offres Similaires

In [ ]:
# Prendre la première recommandation et trouver des offres similaires
reference_job_id = recommendations[0]['job_id']
reference_job = recommender.get_job_details(reference_job_id)

print(f"📌 Offre de référence:")
print(f"   {reference_job['title']} - {reference_job['company']}")
print(f"   Compétences: {', '.join(reference_job['skills'][:8])}")

In [ ]:
# Trouver des offres similaires
similar_jobs = recommender.get_similar_jobs(reference_job_id, top_k=5)

print(f"\n🔗 Offres similaires:\n")
for i, job in enumerate(similar_jobs, 1):
    print(f"{i}. {job['title']} - {job['company']}")
    print(f"   📍 {job['location']}")
    print(f"   Similarité: {job['similarity_score']:.3f}")
    print(f"   Compétences: {', '.join(job['skills'][:5])}")
    print()

## 6. Test avec Upload de CV (Simulation)

In [ ]:
# Créer un CV fictif pour tester
fake_cv_text = """
John Doe
Data Analyst | Machine Learning Engineer
Paris, France
john.doe@email.com

EXPÉRIENCE PROFESSIONNELLE

Senior Data Analyst - Tech Corp (2021-2024)
- Analyse de données massives avec SQL, Python (Pandas, NumPy)
- Création de dashboards Power BI et Tableau
- Développement de modèles prédictifs avec scikit-learn
- Collaboration avec les équipes business pour identifier les KPIs

Data Analyst Junior - StartUp Inc (2019-2021)
- Analyse exploratoire de données (EDA)
- Reporting automatisé avec Python et Excel
- Visualisations interactives

COMPÉTENCES TECHNIQUES
- Langages: Python, SQL, R
- BI Tools: Power BI, Tableau, Looker
- Databases: PostgreSQL, MySQL, MongoDB
- Cloud: AWS (S3, EC2), Azure
- Machine Learning: scikit-learn, XGBoost
- Libraries: Pandas, NumPy, Matplotlib, Seaborn

FORMATION
Master Data Science - Université Paris (2019)
Licence Mathématiques Appliquées - Université Lyon (2017)
"""

# Simuler une recommandation basée sur CV
cv_recommendations = recommender.recommend(
    candidate_profile="",
    cv_text=fake_cv_text,
    keywords=['Power BI', 'Tableau', 'SQL'],
    location_preference='Paris',
    top_k=5
)

print("📄 Recommandations basées sur le CV:\n")
for i, rec in enumerate(cv_recommendations, 1):
    print(f"{i}. {rec['title']} - {rec['company']}")
    print(f"   Score: {rec['score']:.3f}")
    print()

## 7. Analyse des Compétences dans les Recommandations

In [ ]:
# Extraire toutes les compétences des recommandations
all_skills = []
for rec in recommendations:
    all_skills.extend(rec['skills'])

# Compter les occurrences
from collections import Counter
skill_counts = Counter(all_skills)

# Top 15 compétences
top_15_skills = skill_counts.most_common(15)
skills_analysis_df = pd.DataFrame(top_15_skills, columns=['Compétence', 'Occurrences'])

# Visualisation
plt.figure(figsize=(12, 6))
sns.barplot(data=skills_analysis_df, y='Compétence', x='Occurrences', palette='rocket')
plt.title('Top 15 Compétences dans les Recommandations', fontsize=16, fontweight='bold')
plt.xlabel('Nombre d\'occurrences', fontsize=12)
plt.ylabel('Compétence', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Export des Résultats

In [ ]:
# Exporter les recommandations en CSV
export_df = pd.DataFrame([{
    'Rang': i,
    'Titre': rec['title'],
    'Entreprise': rec['company'],
    'Localisation': rec['location'],
    'Type_Contrat': rec['contract_type'],
    'Score_Final': rec['score'],
    'Similarite_Semantique': rec['semantic_similarity'],
    'Skills_Matches': rec['skills_match_count'],
    'Skills_Ratio': rec['skills_match_ratio'],
    'Competences': ', '.join(rec['skills'][:10]),
    'URL': rec['job_url']
} for i, rec in enumerate(recommendations, 1)])

# Sauvegarder
export_df.to_csv('mes_recommandations.csv', index=False, encoding='utf-8-sig')
print("✅ Résultats exportés vers 'mes_recommandations.csv'")

export_df

## 🎉 Conclusion

Ce notebook a démontré les capacités du système de recommandation:

1. ✅ Chargement de 200K+ offres d'emploi
2. ✅ Recherche sémantique intelligente avec Sentence-BERT  
3. ✅ Matching de compétences avec extraction NER
4. ✅ Scoring multi-critères (sémantique + skills + localisation + etc.)
5. ✅ Recherche d'offres similaires
6. ✅ Support de CV (parsing automatique)
7. ✅ Visualisations et statistiques

### Prochaines Étapes

- 🌐 Utiliser l'interface Streamlit: `streamlit run app.py`
- 🔌 Tester l'API FastAPI: `python api.py`
- 📊 Intégrer avec Power BI
- 🚀 Déployer en production
